# Stage 3: Context-Aware Training with RoPE

Standalone notebook for Stage 3 context-aware fine-tuning with Rotary Position Embeddings.

**Prerequisites**: `final_stage2_model.pth` uploaded to `/content/` on Colab (Stage 2 pretrained weights).

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    #print(f"Memory: {props.total_mem / 1024**3:.1f} GB")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")


In [ ]:
# Install core dependencies
# Only install and restart if transformers is not the correct version
import importlib, subprocess, sys

_need_restart = False
try:
    import transformers
    if transformers.__version__ != "4.48.0":
        raise ImportError("wrong version")
    print(f"transformers {transformers.__version__} already installed, skipping.")
except (ImportError, AttributeError):
    _need_restart = True
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "transformers"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.48.0"])

# Install other deps (idempotent, won't trigger restart)
!pip install -q marisa-trie==1.2.1
!pip install -q git+https://github.com/csebuetnlp/normalizer@d80c3c4#egg=normalizer
!pip install -q albumentations==2.0.5

if _need_restart:
    print("\n--- Restarting runtime to load new transformers version ---")
    print("After restart, re-run this cell (it will skip the install).")
    import os
    os.kill(os.getpid(), 9)
else:
    print("All dependencies ready, no restart needed.")

In [ ]:
import subprocess, sys, os

try:
    import transformers
    ver = transformers.__version__
    if ver != '4.48.0':
        raise ImportError(f"Wrong version: {ver}")
    print(f"transformers {ver} already installed, skipping restart")
except (ImportError, AttributeError):
    print("Installing dependencies...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'transformers==4.48.0', 'pdf2image', 'Levenshtein', 'datasets'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/csebuetnlp/normalizer@d80c3c4'])
    print("Dependencies installed. Restarting runtime...")
    os.kill(os.getpid(), 9)


In [ ]:
import os, sys

REPO_ROOT = '/content/GraDeT-HTR'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/shyan23/GraDeT-HTR_context.git {REPO_ROOT}
    print("Repository cloned")
else:
    print("Repository already exists")

sys.path.insert(0, os.path.join(REPO_ROOT, 'GraDeT_HTR'))
os.chdir(os.path.join(REPO_ROOT, 'GraDeT_HTR'))
print(f"Working directory: {os.getcwd()}")


In [ ]:
import gdown, os

DATA_DIR = '/content/training_data'
if not os.path.exists(DATA_DIR) or len(os.listdir(DATA_DIR)) == 0:
    os.makedirs(DATA_DIR, exist_ok=True)
    gdown.download(id='1txYG1JoP62RWWrn_0FxCQ92g9pvYNFWl', output='/content/training_data.zip')
    !unzip -q /content/training_data.zip -d {DATA_DIR}
    print("Data downloaded and extracted")
else:
    print("Data already exists")

# Discover data structure (search recursively for images/ and json/ dirs)
import glob

IMAGES_DIR = None
JSON_DIR = None

for root, dirs, files in os.walk(DATA_DIR):
    for d in dirs:
        full = os.path.join(root, d)
        if d == 'images' and IMAGES_DIR is None:
            IMAGES_DIR = full
        elif d == 'json' and JSON_DIR is None:
            JSON_DIR = full

print(f"IMAGES_DIR: {IMAGES_DIR}")
print(f"JSON_DIR: {JSON_DIR}")
assert IMAGES_DIR is not None, "Could not find images directory!"
assert JSON_DIR is not None, "Could not find JSON directory!"

print(f"Images: {len(os.listdir(IMAGES_DIR))} files")
print(f"JSON:   {len(os.listdir(JSON_DIR))} files")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_OUTPUT_DIR = '/content/drive/MyDrive/GraDeT_HTR_Stage3'
os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)

LOCAL_OUTPUT_DIR = os.path.join(REPO_ROOT, 'output')
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

print(f"Local saves:  {LOCAL_OUTPUT_DIR}")
print(f"GDrive saves: {GDRIVE_OUTPUT_DIR}")


## Fix processor.py

Overwrite processor.py to ensure empty context still gets a fixed-length padding block (prevents collation crashes).

In [ ]:
proc_path = os.path.join(REPO_ROOT, 'GraDeT_HTR', 'processor.py')
with open(proc_path, 'w') as f:
    f.write('''from transformers import GPT2Tokenizer, AutoImageProcessor
from bntokenizer import BnGraphemizerProcessor

from PIL import Image
from typing import List, Union
from config import DTrOCRConfig
from data import DTrOCRProcessorOutput


class DTrOCRProcessor:
    def __init__(self, config: DTrOCRConfig, add_bos_token: bool = False, add_eos_token: bool = False):
        self.vit_processor = AutoImageProcessor.from_pretrained(
            config.vit_hf_model,
            size={
                "height": config.image_size[0],
                "width": config.image_size[1]
            },
            use_fast=True
        )
        self.max_context_length = getattr(config, "max_context_length", 20)
        self.tokeniser = BnGraphemizerProcessor(
            config.bn_vocab_file,
            add_bos_token=add_bos_token,
            add_eos_token=add_eos_token,
            model_max_length=config.max_position_embeddings - int(
                (config.image_size[0] / config.patch_size[0]) * (config.image_size[1] / config.patch_size[1])
            )
        )

    def __call__(
        self,
        images: Union[Image.Image, List[Image.Image]] = None,
        texts: Union[str, List[str]] = None,
        context_text: Union[str, List[str]] = None,
        return_labels: bool = False,
        input_data_format: str = "channels_last",
        padding: Union[bool, str] = False,
        *args,
        **kwargs
    ) -> DTrOCRProcessorOutput:
        import torch

        context_length = 0

        text_inputs = self.tokeniser(
            texts, padding=padding
        ) if texts is not None else None

        if context_text is not None and text_inputs is not None:
            pad_token_id = self.tokeniser.pad_token_id

            if context_text != "":
                context_inputs = self.tokeniser(context_text, padding=False)
                sep_token_id = self.tokeniser.eos_token_id
                context_ids = context_inputs["input_ids"]

                if isinstance(context_ids, list):
                    if len(context_ids) > 0 and isinstance(context_ids[0], list):
                        context_ids = context_ids[0]
                elif hasattr(context_ids, "tolist"):
                    context_ids = context_ids.tolist()
                    if isinstance(context_ids[0], list):
                        context_ids = context_ids[0]

                context_ids = context_ids[:self.max_context_length - 1]
                context_with_sep = context_ids + [sep_token_id]

                actual_len = len(context_with_sep)
                pad_needed = self.max_context_length - actual_len
                context_padded = context_with_sep + [pad_token_id] * pad_needed
                context_attn_mask = [1] * actual_len + [0] * pad_needed
            else:
                context_padded = [pad_token_id] * self.max_context_length
                context_attn_mask = [0] * self.max_context_length

            context_length = self.max_context_length

            target_ids = text_inputs["input_ids"]
            if hasattr(target_ids, "tolist"):
                target_ids = target_ids.tolist()
                if isinstance(target_ids[0], list):
                    target_ids = target_ids[0]
            elif isinstance(target_ids, list) and len(target_ids) > 0 and isinstance(target_ids[0], list):
                target_ids = target_ids[0]

            combined_ids = context_padded + target_ids
            text_inputs["input_ids"] = torch.tensor([combined_ids])

            target_mask = text_inputs["attention_mask"]
            if hasattr(target_mask, "tolist"):
                target_mask = target_mask.tolist()
                if isinstance(target_mask[0], list):
                    target_mask = target_mask[0]
            elif isinstance(target_mask, list) and len(target_mask) > 0 and isinstance(target_mask[0], list):
                target_mask = target_mask[0]

            combined_mask = context_attn_mask + target_mask
            text_inputs["attention_mask"] = torch.tensor([combined_mask])

        image_inputs = self.vit_processor(
            images, input_data_format=input_data_format, *args, **kwargs
        ) if images is not None else None

        return DTrOCRProcessorOutput(
            pixel_values=image_inputs["pixel_values"] if images is not None else None,
            input_ids=text_inputs["input_ids"] if texts is not None else None,
            attention_mask=text_inputs["attention_mask"] if texts is not None else None,
            labels=text_inputs["input_ids"] if texts is not None and return_labels else None,
            context_length=context_length
        )
''')
print("processor.py overwritten with fixed version")

# --- Also fix config.py: add use_rope parameter ---
config_path = os.path.join(REPO_ROOT, 'GraDeT_HTR', 'config.py')
with open(config_path, 'w') as f:
    f.write('''from typing import Optional, Union, Tuple, List, Literal


class DTrOCRConfig:
    def __init__(
        self,
        gpt2_hf_model: str = 'openai-community/gpt2',
        vit_hf_model: str = 'google/vit-base-patch16-224',
        vocab_size: Optional[int] = 1420,
        max_position_embeddings: Optional[int] = 256,
        hidden_size: Optional[int] = 768,
        num_hidden_layers: Optional[int] = 12,
        num_attention_heads: Optional[int] = 12,
        patch_size: Optional[Union[Tuple[int], List[int]]] = (4, 8),
        image_size: Optional[Union[Tuple[int], List[int]]] = (32, 128),
        num_channels: Optional[int] = 3,
        resid_pdrop: Optional[float] = 0.1,
        embd_pdrop: Optional[float] = 0.1,
        attn_pdrop: Optional[float] = 0.1,
        layer_norm_epsilon: Optional[float] = 1e-5,
        attn_implementation: Literal["sdpa", "flash_attention_2"] = "sdpa",
        bn_vocab_file: str = "../tokenization/bn_grapheme_1296_from_bengali.ai.buet.txt",
        max_context_length: Optional[int] = 20,
        use_rope: bool = False,
    ):
        self.gpt2_hf_model = gpt2_hf_model
        self.vit_hf_model = vit_hf_model
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.patch_size = patch_size
        self.image_size = image_size
        self.num_channels = num_channels
        self.vocab_size = vocab_size
        self.max_context_length = max_context_length
        self.max_position_embeddings = max_position_embeddings
        self.resid_pdrop = resid_pdrop
        self.embd_pdrop = embd_pdrop
        self.attn_pdrop = attn_pdrop
        self.layer_norm_epsilon = layer_norm_epsilon
        self._attn_implementation = attn_implementation
        self.bn_vocab_file = bn_vocab_file
        self.use_rope = use_rope

        # other GPT2 config values
        self.n_inner = None
        self.scale_attn_weights = True
        self.scale_attn_by_inverse_layer_idx = False
        self.reorder_and_upcast_attn = False
        self.add_cross_attention = False
        self.activation_function = "gelu_new"
''')
print("config.py overwritten with use_rope support")

# Reload modules
import importlib
import config, processor, data, dataset
importlib.reload(config)
importlib.reload(processor)
importlib.reload(data)
importlib.reload(dataset)
print("Modules reloaded")


## RoPE Implementation

Rotary Position Embedding from the RoFormer paper (Su et al., 2023).

**Key equations:**
- $\theta_i = 10000^{-2(i-1)/d}$ (Eq. 15)
- $R \cdot x = x \odot \cos(m\theta) + \text{rotate\_half}(x) \odot \sin(m\theta)$ (Eq. 34)

In [ ]:
import torch
import torch.nn as nn
import math

class RotaryEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE) from RoFormer paper.
    
    Precomputes cos/sin tables for all positions up to max_position_embeddings.
    Applied to query and key tensors in attention to encode relative positions.
    """
    def __init__(self, dim, max_position_embeddings=256, base=10000.0):
        super().__init__()
        # theta_i = base^(-2(i-1)/d) for i in [1, ..., d/2]
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_seq_len_cached = max_position_embeddings
        
        # Precompute cos/sin for all positions
        t = torch.arange(self.max_seq_len_cached).float()
        freqs = torch.outer(t, self.inv_freq)  # [max_len, dim/2]
        emb = torch.cat((freqs, freqs), dim=-1)  # [max_len, dim]
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())
    
    def forward(self, seq_len):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def rotate_half(x):
    """Rotate half of the hidden dims: [-x2, x1]."""
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    """Apply RoPE to query and key tensors (Eq. 34 from paper)."""
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

print("RoPE core defined: RotaryEmbedding, rotate_half, apply_rotary_pos_emb")


In [ ]:
class RoPEAttention(nn.Module):
    """Multi-head attention with Rotary Position Embeddings.
    
    Replaces GPT2Attention's combined c_attn Conv1D with separate
    q_proj, k_proj, v_proj Linear layers. RoPE is applied to Q and K.
    """
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.hidden_size = config.hidden_size
        
        # Separate projections (instead of GPT2's combined c_attn Conv1D)
        self.q_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.k_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.v_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.c_proj = nn.Linear(config.hidden_size, config.hidden_size)
        
        self.resid_dropout = nn.Dropout(config.resid_pdrop)
        self.rotary_emb = RotaryEmbedding(self.head_dim, config.max_position_embeddings)
    
    def forward(self, hidden_states, attention_mask=None, layer_past=None, use_cache=False):
        batch_size, seq_len, _ = hidden_states.shape
        
        # Project to Q, K, V and reshape to [batch, heads, seq, head_dim]
        q = self.q_proj(hidden_states).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Compute position offset for KV cache
        kv_seq_len = seq_len
        if layer_past is not None:
            kv_seq_len += layer_past[0].shape[-2]
        
        # Get cos/sin for RoPE
        cos_full, sin_full = self.rotary_emb(kv_seq_len)
        if layer_past is not None:
            offset = layer_past[0].shape[-2]
            cos_q = cos_full[offset:offset+seq_len].unsqueeze(0).unsqueeze(0)
            sin_q = sin_full[offset:offset+seq_len].unsqueeze(0).unsqueeze(0)
        else:
            cos_q = cos_full[:seq_len].unsqueeze(0).unsqueeze(0)
            sin_q = sin_full[:seq_len].unsqueeze(0).unsqueeze(0)
        
        # Apply RoPE to Q and K
        q, k = apply_rotary_pos_emb(q, k, cos_q, sin_q)
        
        # Concatenate with past KV cache
        if layer_past is not None:
            past_key, past_value = layer_past
            k = torch.cat([past_key, k], dim=-2)
            v = torch.cat([past_value, v], dim=-2)
        
        present = (k, v) if use_cache else None
        
        # Scaled dot-product attention (uses FlashAttention when available)
        attn_output = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, attn_mask=attention_mask
        )
        
        # Reshape and project output
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        attn_output = self.c_proj(attn_output)
        attn_output = self.resid_dropout(attn_output)
        
        return attn_output, present


class RoPEBlock(nn.Module):
    """Transformer block with RoPE attention (replaces GPT2Block).
    
    Pre-LN architecture: LayerNorm before attention/MLP with residual connections.
    """
    def __init__(self, config, layer_idx=None):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_epsilon)
        self.attn = RoPEAttention(config)
        self.ln_2 = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_epsilon)
        
        inner_dim = config.n_inner if config.n_inner is not None else 4 * config.hidden_size
        self.mlp = nn.Sequential(
            nn.Linear(config.hidden_size, inner_dim),
            nn.GELU(approximate='tanh'),  # Matches GPT2's gelu_new
            nn.Linear(inner_dim, config.hidden_size),
            nn.Dropout(config.resid_pdrop),
        )
    
    def forward(self, hidden_states, attention_mask=None, layer_past=None, use_cache=False, **kwargs):
        # Pre-LN + Attention + Residual
        residual = hidden_states
        hidden_states = self.ln_1(hidden_states)
        attn_output, present = self.attn(
            hidden_states, attention_mask=attention_mask,
            layer_past=layer_past, use_cache=use_cache
        )
        hidden_states = residual + attn_output
        
        # Pre-LN + MLP + Residual
        residual = hidden_states
        hidden_states = self.ln_2(hidden_states)
        hidden_states = residual + self.mlp(hidden_states)
        
        outputs = (hidden_states,)
        if use_cache:
            outputs += (present,)
        return outputs

print("RoPEAttention and RoPEBlock defined")


## RoPE Model Architecture

DTrOCRModelRoPE replaces absolute positional embeddings with RoPE.
DTrOCRLMHeadModelRoPE adds the language modeling head for training.

In [ ]:
from transformers.models.vit.modeling_vit import ViTPatchEmbeddings
from transformers.modeling_attn_mask_utils import _prepare_4d_causal_attention_mask_for_sdpa
from data import DTrOCRModelOutput, DTrOCRLMHeadModelOutput, DTrOCRProcessorOutput
from config import DTrOCRConfig


class DTrOCRModelRoPE(nn.Module):
    """Vision-language transformer backbone with RoPE (no absolute positional embeddings)."""
    
    def __init__(self, config: DTrOCRConfig):
        super().__init__()
        self.patch_embeddings = ViTPatchEmbeddings(config)
        self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        # NO positional_embedding - RoPE handles position encoding inside attention
        
        self.hidden_layers = nn.ModuleList([
            RoPEBlock(config, layer_idx=i) for i in range(config.num_hidden_layers)
        ])
        
        self.dropout = nn.Dropout(config.attn_pdrop)
        self.layer_norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_epsilon)
        self._attn_implementation = config._attn_implementation
    
    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.LongTensor,
        position_ids=None,
        past_key_values=None,
        attention_mask=None,
        use_cache=False,
        context_length=0,
    ) -> DTrOCRModelOutput:
        device = input_ids.device
        input_ids = input_ids.view(-1, input_ids.shape[-1])
        
        if past_key_values is None:
            past_length = 0
            past_key_values = tuple([None] * len(self.hidden_layers))
        else:
            past_length = past_key_values[0][0].size(-2)
        
        patch_embeddings = self.patch_embeddings(pixel_values) if past_length == 0 else None
        token_embeddings = self.token_embedding(input_ids)
        
        # Reorder: [context, SEP, IMAGE, BOS, target]
        if patch_embeddings is not None and context_length > 0:
            context_emb = token_embeddings[:, :context_length, :]
            target_emb = token_embeddings[:, context_length:, :]
            hidden_states = torch.concat([context_emb, patch_embeddings, target_emb], dim=-2)
        elif patch_embeddings is not None:
            hidden_states = torch.concat([patch_embeddings, token_embeddings], dim=-2)
        else:
            hidden_states = token_embeddings
        
        # NO positional embeddings added - RoPE handles this in attention
        hidden_states = self.dropout(hidden_states)
        
        # Build 4D causal attention mask
        if attention_mask is not None:
            if context_length > 0 and patch_embeddings is not None:
                ctx_mask = attention_mask[:, :context_length]
                tgt_mask = attention_mask[:, context_length:]
                patch_mask = torch.ones(
                    attention_mask.shape[0], patch_embeddings.shape[-2],
                    dtype=attention_mask.dtype, device=device
                )
                attention_mask = torch.concat([ctx_mask, patch_mask, tgt_mask], dim=-1)
            else:
                num_patches = patch_embeddings.shape[-2] if patch_embeddings is not None else past_length
                attention_mask = torch.concat([
                    torch.ones(attention_mask.shape[0], num_patches,
                               dtype=attention_mask.dtype, device=device),
                    attention_mask
                ], dim=-1)
            
            attention_mask = _prepare_4d_causal_attention_mask_for_sdpa(
                attention_mask=attention_mask,
                input_shape=(hidden_states.shape[0], hidden_states.shape[1]),
                inputs_embeds=hidden_states,
                past_key_values_length=past_length,
            )
        
        presents = () if use_cache else None
        for layer, layer_past in zip(self.hidden_layers, past_key_values):
            outputs = layer(
                hidden_states, layer_past=layer_past,
                attention_mask=attention_mask, use_cache=use_cache
            )
            hidden_states = outputs[0]
            if use_cache:
                presents += (outputs[1],)
        
        hidden_states = self.layer_norm(hidden_states)
        return DTrOCRModelOutput(hidden_states=hidden_states, past_key_values=presents)

print("DTrOCRModelRoPE defined")


In [ ]:
class DTrOCRLMHeadModelRoPE(nn.Module):
    """Language model head on top of DTrOCRModelRoPE for autoregressive generation."""
    
    def __init__(self, config: DTrOCRConfig):
        super().__init__()
        self.config = config
        self.transformer = DTrOCRModelRoPE(config)
        self.language_model_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        
        image_size, patch_size = config.image_size, config.patch_size
        self.image_embedding_length = int(
            (image_size[0] / patch_size[0]) * (image_size[1] / patch_size[1])
        )
    
    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.LongTensor,
        past_key_values=None,
        position_ids=None,
        attention_mask=None,
        use_cache=False,
        labels=None,
        context_length=0,
        return_per_sample_loss=False,
    ) -> DTrOCRLMHeadModelOutput:
        transformer_output = self.transformer(
            pixel_values=pixel_values,
            input_ids=input_ids,
            past_key_values=past_key_values,
            position_ids=position_ids,
            attention_mask=attention_mask,
            use_cache=use_cache,
            context_length=context_length
        )
        logits = self.language_model_head(transformer_output.hidden_states)
        
        loss, accuracy, per_sample_loss = None, None, None
        if labels is not None:
            labels = labels.to(logits.device)
            
            skip_positions = context_length + self.image_embedding_length
            shift_logits = logits[..., skip_positions:-1, :].contiguous()
            shift_labels = labels[..., context_length + 1:].contiguous()
            
            batch_size = shift_logits.size(0)
            seq_len = shift_logits.size(1)
            
            loss_fct = nn.CrossEntropyLoss(reduction="none")
            loss_per_token = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )
            
            label_matches = shift_labels.view(-1) == torch.argmax(
                torch.nn.functional.softmax(
                    shift_logits.view(-1, shift_logits.size(-1)), dim=-1
                ), dim=-1
            )
            
            loss_2d = loss_per_token.view(batch_size, seq_len)
            matches_2d = label_matches.view(batch_size, seq_len)
            
            if attention_mask is not None:
                mask = attention_mask[..., context_length + 1:].reshape(batch_size, seq_len)
                per_sample_loss = (mask * loss_2d).sum(dim=1) / mask.sum(dim=1)
                loss = per_sample_loss.mean()
                accuracy = (mask * matches_2d.float()).sum() / mask.sum()
            else:
                per_sample_loss = loss_2d.mean(dim=1)
                loss = per_sample_loss.mean()
                accuracy = matches_2d.float().mean()
        
        return DTrOCRLMHeadModelOutput(
            loss=loss,
            logits=logits,
            accuracy=accuracy,
            past_key_values=transformer_output.past_key_values,
            per_sample_loss=per_sample_loss if return_per_sample_loss else None
        )

print("DTrOCRLMHeadModelRoPE defined")
print(f"  Image embedding length: {int((32/4) * (128/8))} patches")


## Configuration & Data

In [ ]:
from config import DTrOCRConfig

VOCAB_FILE = os.path.join(REPO_ROOT, 'tokenization', 'bn_grapheme_1296_from_bengali.ai.buet.txt')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = DTrOCRConfig(
    bn_vocab_file=VOCAB_FILE,
    use_rope=True,
    max_context_length=20,
)

# Stage 3 hyperparameters
LEARNING_RATE = 5e-6
BATCH_SIZE = 96
EPOCHS = 7
LAMBDA_MARGIN = 0.7

print(f"Device: {device}")
print(f"Config: RoPE={config.use_rope}, context_len={config.max_context_length}")
print(f"LR={LEARNING_RATE}, batch={BATCH_SIZE}, epochs={EPOCHS}, lambda={LAMBDA_MARGIN}")


In [ ]:
from dataset import split_context_data

train_dataset, test_dataset = split_context_data(
    IMAGES_DIR, JSON_DIR, config, test_size=0.05
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"Train: {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"Test:  {len(test_dataset)} samples ({len(test_loader)} batches)")

# Verify a sample
sample = train_dataset[0]
print(f"\nSample shapes:")
print(f"  pixel_values:    {sample['pixel_values'].shape}")
print(f"  input_ids:       {sample['input_ids'].shape}")
print(f"  input_ids_iso:   {sample['input_ids_isolated'].shape}")
print(f"  context_length:  {sample['context_length']}")
print(f"  text: {sample['text']}")
print(f"  prev_text: '{sample['prev_text']}'")


## Weight Conversion & Loading

Convert Stage 2 GPT2Block weights to RoPEBlock format:
- Conv1D weights transposed to Linear format
- Combined `c_attn` split into separate `q_proj`, `k_proj`, `v_proj`
- `positional_embedding` dropped (RoPE replaces it)
- MLP Conv1D (`c_fc`, `c_proj`) mapped to Sequential indices

In [ ]:
def convert_gpt2_to_rope(state_dict):
    """Convert GPT2Block state_dict to RoPEBlock format."""
    new_state_dict = {}
    skipped = []
    converted = []
    
    for key, value in state_dict.items():
        clean_key = key.replace("_orig_mod.", "")
        
        # Skip absolute positional embeddings
        if "positional_embedding" in clean_key:
            skipped.append(clean_key)
            continue
        
        # Skip GPT2's causal mask buffers
        if "attn.bias" in clean_key or "attn.masked_bias" in clean_key:
            skipped.append(clean_key)
            continue
        
        # Combined c_attn Conv1D -> separate q/k/v Linear
        if "attn.c_attn.weight" in clean_key:
            prefix = clean_key.replace("attn.c_attn.weight", "")
            w = value.t()  # Conv1D (in, 3*out) -> Linear (3*out, in)
            h = w.shape[0] // 3
            new_state_dict[prefix + "attn.q_proj.weight"] = w[:h]
            new_state_dict[prefix + "attn.k_proj.weight"] = w[h:2*h]
            new_state_dict[prefix + "attn.v_proj.weight"] = w[2*h:]
            converted.append(f"{clean_key} -> q/k/v_proj.weight")
        elif "attn.c_attn.bias" in clean_key:
            prefix = clean_key.replace("attn.c_attn.bias", "")
            h = value.shape[0] // 3
            new_state_dict[prefix + "attn.q_proj.bias"] = value[:h]
            new_state_dict[prefix + "attn.k_proj.bias"] = value[h:2*h]
            new_state_dict[prefix + "attn.v_proj.bias"] = value[2*h:]
            converted.append(f"{clean_key} -> q/k/v_proj.bias")
        
        # Output projection Conv1D -> Linear
        elif "attn.c_proj.weight" in clean_key:
            new_state_dict[clean_key] = value.t()
            converted.append(f"{clean_key} (transposed)")
        
        # MLP Conv1D -> Sequential Linear
        elif "mlp.c_fc.weight" in clean_key:
            prefix = clean_key.replace("mlp.c_fc.weight", "")
            new_state_dict[prefix + "mlp.0.weight"] = value.t()
            converted.append(f"{clean_key} -> mlp.0.weight")
        elif "mlp.c_fc.bias" in clean_key:
            prefix = clean_key.replace("mlp.c_fc.bias", "")
            new_state_dict[prefix + "mlp.0.bias"] = value
        elif "mlp.c_proj.weight" in clean_key:
            prefix = clean_key.replace("mlp.c_proj.weight", "")
            new_state_dict[prefix + "mlp.2.weight"] = value.t()
            converted.append(f"{clean_key} -> mlp.2.weight")
        elif "mlp.c_proj.bias" in clean_key:
            prefix = clean_key.replace("mlp.c_proj.bias", "")
            new_state_dict[prefix + "mlp.2.bias"] = value
        
        # Everything else passes through unchanged
        else:
            new_state_dict[clean_key] = value
    
    print(f"Converted {len(converted)} weight groups:")
    for c in converted[:5]:
        print(f"  {c}")
    if len(converted) > 5:
        print(f"  ... and {len(converted) - 5} more")
    print(f"Skipped {len(skipped)} keys: {skipped}")
    
    return new_state_dict


# Load Stage 2 weights
print("Loading Stage 2 model (final_stage2_model.pth)...")
stage2_path = '/content/final_stage2_model.pth'
stage2_state = torch.load(stage2_path, map_location='cpu')
print(f"Loaded {len(stage2_state)} parameters")

# Build RoPE model
model = DTrOCRLMHeadModelRoPE(config)
total_params = sum(p.numel() for p in model.parameters())
print(f"RoPE model created: {total_params:,} parameters")

# Convert and load weights
print("\nConverting GPT2 weights to RoPE format...")
rope_state = convert_gpt2_to_rope(stage2_state)

missing, unexpected = model.load_state_dict(rope_state, strict=False)
print(f"\nMissing keys ({len(missing)}):")
for k in missing:
    print(f"  {k}")
print(f"Unexpected keys ({len(unexpected)}):")
for k in unexpected:
    print(f"  {k}")

model = model.to(device)
print(f"\nModel loaded on {device}")


## Training

Context-aware dual-path training with margin penalty:

$$L = -\log(P_c) - \log(P_i) + \lambda \cdot \max(0, \text{loss}_{ctx} - \text{loss}_{iso})$$

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
print(f"Optimizer: AdamW, LR={LEARNING_RATE}, weight_decay=0.01")


In [ ]:
import tqdm

def validate_model(model, dataloader, device):
    """Run validation on the context-aware model (dual-path)."""
    model.train(False)
    ctx_losses, iso_losses, ctx_accs, iso_accs = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm.tqdm(dataloader, desc="Validating"):
            pv = batch['pixel_values'].to(device)
            
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                # Context path
                ctx_out = model(
                    pixel_values=pv,
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['labels'].to(device),
                    context_length=batch['context_length'][0].item()
                        if isinstance(batch['context_length'], torch.Tensor)
                        else batch['context_length'],
                )
                # Isolation path
                iso_out = model(
                    pixel_values=pv,
                    input_ids=batch['input_ids_isolated'].to(device),
                    attention_mask=batch['attention_mask_isolated'].to(device),
                    labels=batch['labels_isolated'].to(device),
                    context_length=batch['context_length_isolated'][0].item()
                        if isinstance(batch['context_length_isolated'], torch.Tensor)
                        else batch['context_length_isolated'],
                )
            
            ctx_losses.append(ctx_out.loss.item())
            iso_losses.append(iso_out.loss.item())
            if ctx_out.accuracy is not None:
                ctx_accs.append(ctx_out.accuracy.item())
            if iso_out.accuracy is not None:
                iso_accs.append(iso_out.accuracy.item())
    
    model.train(True)
    return {
        'ctx_loss': sum(ctx_losses) / len(ctx_losses),
        'iso_loss': sum(iso_losses) / len(iso_losses),
        'ctx_acc': sum(ctx_accs) / len(ctx_accs) if ctx_accs else 0,
        'iso_acc': sum(iso_accs) / len(iso_accs) if iso_accs else 0,
    }

print("validate_model() defined")


In [ ]:
import shutil

train_history = {
    'epoch': [], 'train_loss': [], 'train_ctx_loss': [], 'train_iso_loss': [],
    'val_ctx_loss': [], 'val_iso_loss': [], 'val_ctx_acc': [], 'val_iso_acc': [],
}

model.train(True)
print(f"Starting Stage 3 training: {EPOCHS} epochs")
print(f"Dual-path loss: L = ctx_loss + iso_loss + {LAMBDA_MARGIN} * max(0, ctx_loss - iso_loss)")
print("=" * 70)

for epoch in range(EPOCHS):
    epoch_losses, epoch_ctx, epoch_iso = [], [], []
    model.train(True)
    
    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        pv = batch['pixel_values'].to(device)
        
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            # Context path
            ctx_out = model(
                pixel_values=pv,
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                labels=batch['labels'].to(device),
                context_length=batch['context_length'][0].item()
                    if isinstance(batch['context_length'], torch.Tensor)
                    else batch['context_length'],
            )
            # Isolation path
            iso_out = model(
                pixel_values=pv,
                input_ids=batch['input_ids_isolated'].to(device),
                attention_mask=batch['attention_mask_isolated'].to(device),
                labels=batch['labels_isolated'].to(device),
                context_length=batch['context_length_isolated'][0].item()
                    if isinstance(batch['context_length_isolated'], torch.Tensor)
                    else batch['context_length_isolated'],
            )
            
            # Dual-path loss with margin penalty
            margin = torch.clamp(ctx_out.loss - iso_out.loss, min=0)
            loss = ctx_out.loss + iso_out.loss + LAMBDA_MARGIN * margin
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_losses.append(loss.item())
        epoch_ctx.append(ctx_out.loss.item())
        epoch_iso.append(iso_out.loss.item())
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'ctx': f'{ctx_out.loss.item():.4f}',
            'iso': f'{iso_out.loss.item():.4f}'
        })
    
    # Validation
    val_metrics = validate_model(model, test_loader, device)
    
    # Record history
    avg_loss = sum(epoch_losses) / len(epoch_losses)
    avg_ctx = sum(epoch_ctx) / len(epoch_ctx)
    avg_iso = sum(epoch_iso) / len(epoch_iso)
    
    train_history['epoch'].append(epoch + 1)
    train_history['train_loss'].append(avg_loss)
    train_history['train_ctx_loss'].append(avg_ctx)
    train_history['train_iso_loss'].append(avg_iso)
    train_history['val_ctx_loss'].append(val_metrics['ctx_loss'])
    train_history['val_iso_loss'].append(val_metrics['iso_loss'])
    train_history['val_ctx_acc'].append(val_metrics['ctx_acc'])
    train_history['val_iso_acc'].append(val_metrics['iso_acc'])
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train: loss={avg_loss:.4f}, ctx={avg_ctx:.4f}, iso={avg_iso:.4f}")
    print(f"  Val:   ctx_loss={val_metrics['ctx_loss']:.4f}, iso_loss={val_metrics['iso_loss']:.4f}")
    print(f"         ctx_acc={val_metrics['ctx_acc']:.4f},  iso_acc={val_metrics['iso_acc']:.4f}")
    
    # Save checkpoint (local + GDrive)
    ckpt_name = f'stage3_checkpoint_epoch_{epoch+1}.pt'
    ckpt = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_loss,
        'val_ctx_loss': val_metrics['ctx_loss'],
        'val_iso_loss': val_metrics['iso_loss'],
        'val_ctx_acc': val_metrics['ctx_acc'],
        'val_iso_acc': val_metrics['iso_acc'],
    }
    local_ckpt = os.path.join(LOCAL_OUTPUT_DIR, ckpt_name)
    torch.save(ckpt, local_ckpt)
    shutil.copy2(local_ckpt, os.path.join(GDRIVE_OUTPUT_DIR, ckpt_name))
    print(f"  Saved: {ckpt_name} (local + GDrive)")

print("\n" + "=" * 70)
print("Training complete!")


## Save & Analyze

In [ ]:
import shutil

final_name = 'final_stage3_rope_model.pth'

# Save to CPU for portability
model.to('cpu')
local_final = os.path.join(LOCAL_OUTPUT_DIR, final_name)
torch.save(model.state_dict(), local_final)
shutil.copy2(local_final, os.path.join(GDRIVE_OUTPUT_DIR, final_name))
print(f"Final model saved: {final_name}")
print(f"  Local:  {local_final}")
print(f"  GDrive: {os.path.join(GDRIVE_OUTPUT_DIR, final_name)}")

# Download to local machine
from google.colab import files
files.download(local_final)

# Move model back to GPU for any further use
model.to(device)
print(f"Model back on {device}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training losses
axes[0].plot(train_history['epoch'], train_history['train_loss'], 'b-o', label='Total Loss')
axes[0].plot(train_history['epoch'], train_history['train_ctx_loss'], 'g--s', label='Context Loss')
axes[0].plot(train_history['epoch'], train_history['train_iso_loss'], 'r--^', label='Isolation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation losses
axes[1].plot(train_history['epoch'], train_history['val_ctx_loss'], 'g-o', label='Context Loss')
axes[1].plot(train_history['epoch'], train_history['val_iso_loss'], 'r-o', label='Isolation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Losses')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Validation accuracies
axes[2].plot(train_history['epoch'], train_history['val_ctx_acc'], 'g-o', label='Context Accuracy')
axes[2].plot(train_history['epoch'], train_history['val_iso_acc'], 'r-o', label='Isolation Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Validation Accuracies')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(LOCAL_OUTPUT_DIR, 'stage3_training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
shutil.copy2(plot_path, os.path.join(GDRIVE_OUTPUT_DIR, 'stage3_training_curves.png'))
plt.show()
print("Training curves saved (local + GDrive)")


In [ ]:
print("=" * 60)
print("CONTEXT BENEFIT ANALYSIS")
print("=" * 60)

if len(train_history['val_ctx_loss']) > 0:
    final_ctx_loss = train_history['val_ctx_loss'][-1]
    final_iso_loss = train_history['val_iso_loss'][-1]
    final_ctx_acc = train_history['val_ctx_acc'][-1]
    final_iso_acc = train_history['val_iso_acc'][-1]
    
    loss_improvement = final_iso_loss - final_ctx_loss
    loss_pct = (loss_improvement / final_iso_loss * 100) if final_iso_loss > 0 else 0
    acc_improvement = final_ctx_acc - final_iso_acc
    
    print(f"\nFinal Validation Metrics:")
    print(f"  Context Loss:   {final_ctx_loss:.4f}")
    print(f"  Isolation Loss: {final_iso_loss:.4f}")
    print(f"  Loss Improvement: {loss_improvement:.4f} ({loss_pct:.1f}%)")
    print()
    print(f"  Context Accuracy:   {final_ctx_acc:.4f}")
    print(f"  Isolation Accuracy: {final_iso_acc:.4f}")
    print(f"  Accuracy Improvement: {acc_improvement:.4f}")
    print()
    
    if loss_improvement > 0:
        print("Context HELPS: Model performs better with previous word context.")
    else:
        print("Context does NOT help yet. Consider more training or tuning lambda.")
    
    # Show progression
    print("\nEpoch-by-epoch context benefit (ctx_loss - iso_loss):")
    for i, ep in enumerate(train_history['epoch']):
        diff = train_history['val_ctx_loss'][i] - train_history['val_iso_loss'][i]
        bar = "+" * max(0, int(-diff * 20)) if diff < 0 else "-" * max(0, int(diff * 20))
        print(f"  Epoch {ep:2d}: {diff:+.4f} {bar}")
else:
    print("No validation data recorded.")
